# Aggregation Winner Barplots

This notebook compares matched aggregation triplets (`Ours`, `PCA`, `Additive`) by win rate. A comparison is one composition/task row for image experiments or one model/dataset/composition row for LLM selective generation. If several aggregations tie for the best score, the winner credit is split equally.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mdu.eval.aggregation_winners import (
    AGGREGATION_ORDER,
    build_image_winner_records_from_csv,
    build_llm_winner_records_from_dir,
    empty_winner_records,
    summarize_winners,
)

IMAGE_RESULTS_CSV = ROOT / "resources/refactored/results.csv"
LLM_RESULTS_DIR = ROOT / "resources/llm_resources"
OUTPUT_DIR = ROOT / "resources/paper_tables/aggregation_winner_barplots"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
frames = []

if IMAGE_RESULTS_CSV.exists():
    image_records = build_image_winner_records_from_csv(IMAGE_RESULTS_CSV)
    if image_records.empty:
        print(f"No matched image aggregation triplets found in {IMAGE_RESULTS_CSV}")
    else:
        frames.append(image_records)
else:
    print(f"Image results not found: {IMAGE_RESULTS_CSV}")

if LLM_RESULTS_DIR.exists():
    llm_records = build_llm_winner_records_from_dir(LLM_RESULTS_DIR)
    if llm_records.empty:
        print(f"No matched LLM aggregation triplets found in {LLM_RESULTS_DIR}")
    else:
        frames.append(llm_records)
else:
    print(f"LLM results directory not found: {LLM_RESULTS_DIR}")

winner_records = pd.concat(frames, ignore_index=True) if frames else empty_winner_records()
print(f"Winner records: {len(winner_records)}")
display(winner_records.head())

In [ ]:
PANEL_SPECS = [
    ("overall", None, "All tasks"),
    ("ood_detection", "ood_detection", "OOD detection"),
    ("misclassification_detection", "misclassification_detection", "Misclassification detection"),
    ("selective_prediction", "selective_prediction", "Selective prediction"),
    ("selective_generation", "selective_generation", "Selective generation"),
]

def records_for_problem(problem_type):
    if problem_type is None:
        return winner_records
    return winner_records[winner_records["problem_type"].eq(problem_type)].copy()

summary_tables = {}
for name, problem_type, title in PANEL_SPECS:
    subset = records_for_problem(problem_type)
    summary_tables[name] = summarize_winners(subset)
    print(f"{title}: {len(subset)} rows")
    display(summary_tables[name])

In [ ]:
PALETTE = {
    "Ours": "#0072B2",
    "PCA": "#D55E00",
    "Additive": "#009E73",
}

def plot_win_rate_barplot(records, title, output_path):
    if records.empty:
        print(f"No data for {title}; skipped {output_path.name}")
        return None

    summary = summarize_winners(records)
    values = summary["win_rate"].fillna(0.0) * 100.0
    counts = summary["n_comparisons"].fillna(0).astype(int)
    colors = [PALETTE[aggregation] for aggregation in summary["aggregation"]]

    with plt.rc_context({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": ["DejaVu Serif"],
        "font.size": 7,
        "axes.labelsize": 7,
        "axes.titlesize": 7.5,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "axes.linewidth": 0.6,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }):
        fig, ax = plt.subplots(figsize=(3.35, 2.25), constrained_layout=True)
        bars = ax.bar(
            summary["aggregation"],
            values,
            color=colors,
            edgecolor="white",
            linewidth=0.45,
            zorder=3,
        )

        labels = [f"{value:.1f}%\n(n={count})" for value, count in zip(values, counts)]
        ax.bar_label(bars, labels=labels, padding=2, fontsize=5.8)
        ax.set_ylim(0, 108)
        ax.set_ylabel("Win rate (%)")
        ax.set_title(title)
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, color="0.88", linewidth=0.5)
        ax.xaxis.grid(False)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        for spine in ("left", "bottom"):
            ax.spines[spine].set_linewidth(0.6)
        ax.tick_params(axis="both", width=0.6, length=2.5)
        fig.savefig(output_path, bbox_inches="tight", pad_inches=0.02)

    plt.show()
    return summary

plot_summaries = {}
for name, problem_type, title in PANEL_SPECS:
    subset = records_for_problem(problem_type)
    output_path = OUTPUT_DIR / f"win_rate_{name}.pdf"
    plot_summaries[name] = plot_win_rate_barplot(subset, title, output_path)

print(f"Saved plots to {OUTPUT_DIR}")

In [ ]:
available_summaries = {
    name: summary.assign(panel=name)
    for name, summary in plot_summaries.items()
    if summary is not None
}
if available_summaries:
    display(pd.concat(available_summaries.values(), ignore_index=True))